# AssistantAgent and ConversableAgent

**02. Setup and the AgentChat API** introduced `OpenAIChatCompletionClient` and `await agent.run()`. This notebook goes deep on **`AssistantAgent`**: constructor parameters, **`on_messages`** vs **`run`** / **`run_stream`**, **`reflect_on_tool_use`**, multiple assistant instances, and a **legacy ConversableAgent mapping table** for v0.2 migrants.

We build a **Notes API assistant** with tools over the shared **NOTES** / **DOC_CHUNKS** corpus. Prerequisites: **02**. Next: **04. UserProxyAgent and Human Input**.

In [ ]:
OPENAI_API_KEY = "sk-proj-placeholder-replace-with-your-real-key"

import asyncio
import importlib.metadata as im
import os

os.environ.setdefault("OPENAI_API_KEY", OPENAI_API_KEY)

for pkg in ("autogen-agentchat", "autogen-ext", "autogen-core"):
    try:
        print(f"{pkg}:", im.version(pkg))
    except im.PackageNotFoundError:
        print(f"{pkg}: not installed — pip install autogen-agentchat autogen-ext[openai]")

In [ ]:
NOTES = {
    "n1": "The Notes API is built with FastAPI. Routes live under /notes.",
    "n2": "Run alembic upgrade head after pulling database migrations.",
    "n3": "JWT bearer tokens use Authorization: Bearer header.",
    "n4": "Pytest fixtures for DB setup belong in conftest.py.",
}

DOC_CHUNKS = [
    {"id": "c1", "text": NOTES["n1"]},
    {"id": "c2", "text": NOTES["n2"]},
    {"id": "c3", "text": NOTES["n3"]},
    {"id": "c4", "text": NOTES["n4"]},
    {"id": "c5", "text": "Use httpx.AsyncClient in FastAPI tests for async endpoints."},
]

print("Notes corpus:", list(NOTES.keys()), "| chunks:", [c["id"] for c in DOC_CHUNKS])

---

## 1. AssistantAgent — The Workhorse

`AssistantAgent` is the 0.4 replacement for most v0.2 **`AssistantAgent`** and **`ConversableAgent`** use cases that involve an LLM:

```
messages in ──► model_client (LLM) ──► tool calls? ──► tools ──► reflect? ──► reply out
```

| Parameter | Purpose |
|-----------|--------|
| **`name`** | Agent identity in logs and multi-agent routing |
| **`model_client`** | `OpenAIChatCompletionClient` or compatible |
| **`system_message`** | Role instructions (replaces v0.2 `system_message` on agent) |
| **`tools`** | Async functions the LLM may invoke (**06**) |
| **`reflect_on_tool_use`** | If `True`, model synthesizes after tool results |
| **`handoffs`** | Delegate to other agents in teams (**08**, **11**) |

---

## 2. Legacy ConversableAgent Mapping Table

v0.2 **`ConversableAgent`** was the universal base class. In 0.4, pick the right built-in or subclass **`BaseChatAgent`**:

| v0.2 `ConversableAgent` pattern | AgentChat 0.4 equivalent |
|--------------------------------|--------------------------|
| LLM assistant with system prompt | `AssistantAgent` |
| Human stands in for user | `UserProxyAgent` (**04**) |
| `register_reply` custom logic | Custom `BaseChatAgent` or tool functions |
| `code_execution_config` | `CodeExecutorAgent` + team (**07**) |
| `human_input_mode="ALWAYS"` | `UserProxyAgent(input_func=...)` |
| `chat_messages` dict state | `save_state` / `load_state` (**12**) |
| `GroupChat` participants | `RoundRobinGroupChat` / `SelectorGroupChat` (**08**) |
| `GroupChatManager` | Team class handles speaker order |
| `llm_config` | `model_client` |

There is no drop-in `ConversableAgent` — the 0.4 design favors **typed agents** over one mega base class.

---

## 3. Notes API Tools for the Assistant

Register search tools the assistant can call (full tool patterns in **06**):

In [ ]:
async def get_note(note_id: str) -> str:
    """Fetch a note by id (n1, n2, n3, n4)."""
    return NOTES.get(note_id, f"Unknown note id: {note_id}")


async def search_docs(query: str) -> str:
    """Keyword search over documentation chunks c1-c5."""
    tokens = set(query.lower().split())
    hits = []
    for chunk in DOC_CHUNKS:
        if tokens & set(chunk["text"].lower().split()):
            hits.append(f"[{chunk['id']}] {chunk['text']}")
    return "\n".join(hits) if hits else "No matching chunks."


NOTES_TOOLS = [get_note, search_docs]
print("tools:", [t.__name__ for t in NOTES_TOOLS])

---

## 4. Constructing the Notes API Assistant

In [ ]:
from autogen_agentchat.agents import AssistantAgent
from autogen_ext.models.openai import OpenAIChatCompletionClient

model_client = OpenAIChatCompletionClient(model="gpt-4o-mini", api_key=OPENAI_API_KEY)

notes_assistant = AssistantAgent(
    name="notes_assistant",
    model_client=model_client,
    tools=NOTES_TOOLS,
    system_message=(
        "You are a backend mentor for the Notes API. "
        "Use get_note and search_docs before guessing. Be concise."
    ),
    reflect_on_tool_use=True,
)
print("agent:", notes_assistant.name)

---

## 5. reflect_on_tool_use

| Value | Behavior |
|-------|----------|
| **`True`** (recommended) | Model reads tool output and composes a natural answer |
| **`False`** | Return raw tool result as final message |

v0.2 often required manual follow-up turns; 0.4 integrates reflection in the same `on_messages` call when enabled.

---

## 6. run() — Task-Oriented Entry Point

In [ ]:
async def ask_notes_api() -> None:
    result = await notes_assistant.run(
        task="How do I run database migrations for the Notes API?"
    )
    print(result.messages[-1].content)


await ask_notes_api()

---

## 7. on_messages — Message-Oriented Control

`on_messages` accepts a **`Sequence[BaseChatMessage]`** and returns **`Response`** with **`chat_message`**:

```
run(task="...")  ≈  on_messages([TextMessage(content=task, source="user")])
```

In [ ]:
from autogen_agentchat.messages import TextMessage
from autogen_core import CancellationToken


async def explicit_messages() -> None:
    response = await notes_assistant.on_messages(
        [TextMessage(content="Which note id mentions JWT?", source="user")],
        cancellation_token=CancellationToken(),
    )
    print("chat_message:", response.chat_message.content)
    if response.inner_messages:
        print("inner/tool messages:", len(response.inner_messages))


await explicit_messages()

---

## 8. run_stream vs on_messages_stream

Streaming yields each message as the agent thinks, calls tools, and reflects:

In [ ]:
from autogen_agentchat.ui import Console


async def stream_notes() -> None:
    stream = notes_assistant.run_stream(task="Search docs for FastAPI testing guidance.")
    await Console(stream)


await stream_notes()

Inspect the stream manually:

```python
from autogen_agentchat.base import TaskResult

async for msg in notes_assistant.run_stream(task="..."):
    if isinstance(msg, TaskResult):
        print(msg.stop_reason)
    else:
        print(msg.source, msg.content)
```

---

## 9. Multiple Assistant Instances — Specialist Roles

Share one `model_client`, differentiate with **name** and **system_message**:

In [ ]:
migration_expert = AssistantAgent(
    name="migration_expert",
    model_client=model_client,
    tools=[get_note],
    system_message="You only answer Alembic and database migration questions. Cite note ids.",
)

auth_expert = AssistantAgent(
    name="auth_expert",
    model_client=model_client,
    tools=[get_note, search_docs],
    system_message="You only answer JWT and auth header questions.",
)


async def ask_specialists() -> None:
    r1 = await migration_expert.run(task="What command applies migrations?")
    r2 = await auth_expert.run(task="How are tokens passed?")
    print("migration:", r1.messages[-1].content[:100], "...")
    print("auth:", r2.messages[-1].content[:100], "...")


await ask_specialists()

Combine specialists in a **team** (**05**, **08**) instead of calling them sequentially in application code when you want emergent dialogue.

---

## 10. System Message Design for the Notes API

| Guideline | Example |
|-----------|--------|
| State scope | "Notes API backend only" |
| Mandate tools | "Call search_docs before answering" |
| Output format | "Cite note id in brackets" |
| Refusal | "Say you don't know if tools return nothing" |

Custom role templates: **10. Custom Agent Roles and System Messages**.

---

## 11. ASCII — Single Assistant Turn

```
User task
    │
    ▼
┌─────────────────┐
│ AssistantAgent  │
│  system_message   │
└────────┬────────┘
         │ LLM call
         ▼
    tool call? ──no──► final TextMessage
         │
        yes
         ▼
    run get_note / search_docs
         │
         ▼
 reflect_on_tool_use? ──► LLM synthesizes answer
```

---

## 12. Comparison — AssistantAgent vs LangGraph Agent Node

| | `AssistantAgent` | LangGraph prebuilt agent (**06–07**) |
|--|------------------|--------------------------------------|
| State | Internal to agent run | Central `MessagesState` |
| Tool loop | Inside `on_messages` | Graph cycle LLM ↔ tools |
| Multi-agent | Needs Team | Subgraphs / supervisor (**11**) |
| Visibility | Stream messages | `stream_mode` + graph diagram |

Same LLM capabilities — different orchestration envelope.

---

## 13. v0.2 AssistantAgent Quick Map

| v0.2 | 0.4 AgentChat |
|------|---------------|
| `AssistantAgent(name, llm_config, system_message)` | `AssistantAgent(name, model_client, system_message)` |
| `register_function` | `tools=[fn, ...]` on constructor |
| `agent.generate_reply(...)` | `await agent.on_messages(...)` |
| `agent.initiate_chat(other, ...)` | Use `RoundRobinGroupChat` or call `other` via team (**05**) |

---

## 14. When Not to Use AssistantAgent

- **Pure human relay** → `UserProxyAgent` (**04**)
- **Sandboxed code execution** → `CodeExecutorAgent` (**07**)
- **Custom non-LLM logic** → subclass `BaseChatAgent`
- **Fixed DAG workflows** → LangGraph may be simpler

In [ ]:
await model_client.close()
print("model_client closed")

---

## 15. Learning Path — This Series

```
01 Introduction ──► 02 Setup/AgentChat API ──► 03 AssistantAgent
     │
     ▼
04 UserProxy/HITL ──► 05 Two-agent patterns ──► 06 Tools
     │
     ▼
07 Code execution ──► 08 GroupChat teams ──► 09 Speaker selection
     │
     ▼
10 Custom roles ──► 11 Sequential/hierarchical ──► 12 Memory/state
     │
     ▼
13 Termination/guardrails ──► 14 AutoGen Studio ──► 15 Observability
     │
     ▼
16 Production patterns
```

---

## 16. Common Beginner Mistakes

| Mistake | Consequence | Fix |
|---------|-------------|-----|
| Using legacy `pyautogen` 0.2 imports | Wrong API, broken tutorials | Install `autogen-agentchat` 0.4+ |
| Forgetting `await` on `run()` | Coroutine never executes | Use `async def` + `await` or `asyncio.run()` |
| `reflect_on_tool_use=False` unexpectedly | Raw JSON tool output shown to user | Set `True` for natural answers |
| Duplicate specialists without teams | Higher cost, no dialogue | Use `RoundRobinGroupChat` (**08**) |
| No termination condition on teams | Unbounded LLM spend | `TextMentionTermination` (**13**) |

---

## 17. Summary

`AssistantAgent` is the primary **LLM agent** in AgentChat 0.4 — configure **`name`**, **`model_client`**, **`system_message`**, **`tools`**, and **`reflect_on_tool_use`**. Use **`run`** for tasks, **`on_messages`** for explicit transcripts, **`run_stream`** + **`Console`** for live output.

Key takeaways:

- v0.2 **`ConversableAgent`** splits into **`AssistantAgent`**, **`UserProxyAgent`**, teams, and custom agents.
- **`reflect_on_tool_use=True`** lets the model answer after tool results in one turn.
- Multiple specialists share a **model client** but need distinct **system messages**.
- Notes API **NOTES** / **DOC_CHUNKS** power realistic tool-calling demos.
- Teams compose assistants into multi-agent flows (**05**, **08**).

Next: **04. UserProxyAgent and Human Input** — `input_func`, approval loops, and comparison to LangGraph HITL.